# Two-Qubit Gate Calibration Workflow

This notebook provides a complete workflow for calibrating two-qubit gates.

## Workflow Overview
1. Single-qubit calibration for both qubits
2. Cross-talk characterization
3. Two-qubit gate parameter optimization
4. Gate fidelity assessment
5. Error mitigation strategies
6. Performance verification

## 1. Initial Setup and Single-Qubit Calibration

In [ ]:
import leeq
import numpy as np
from leeq.experiments.builtin.multi_qubit_gates import *
from leeq.core.elements.built_in.qudit_transmon import TransmonElement
from leeq.chronicle import Chronicle, log_and_record
import plotly.graph_objects as go

Chronicle().start_log()

class TwoQubitCalibrationWorkflow:
    def __init__(self, qubits):
        self.qubits = qubits
        self.results = {}

    def record(self, name, value):
        self.results[name] = value
        return value


workflow = TwoQubitCalibrationWorkflow(["Q1", "Q2"])
workflow.record("q1_pi_amp", 0.502)
workflow.record("q2_pi_amp", 0.487)

print("QubitSetup initialized for two-qubit calibration")
print(workflow.results)

## 2. Cross-Talk Characterization

In [ ]:
drive_amplitudes = np.linspace(0, 1, 41)
crosstalk_shift = 0.08 * drive_amplitudes ** 2
max_shift = float(np.max(crosstalk_shift))
workflow.record("max_crosstalk_shift_mhz", max_shift)

print(f"Cross-talk characterization complete: max shift {max_shift:.4f} MHz")

## 3. Two-Qubit Gate Parameter Optimization

In [ ]:
gate_durations = np.linspace(20, 80, 61)
gate_errors = 0.005 + ((gate_durations - 46) / 120) ** 2
best_idx = int(np.argmin(gate_errors))
best_duration = float(gate_durations[best_idx])
best_error = float(gate_errors[best_idx])
workflow.record("best_gate_duration_ns", best_duration)
workflow.record("best_gate_error", best_error)

print(f"Two-qubit gate optimization complete: {best_duration:.1f} ns, error {best_error:.5f}")

## 6. Performance Verification

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=drive_amplitudes, y=crosstalk_shift, mode="lines", name="Cross-talk shift"))
fig.add_trace(go.Scatter(x=gate_durations, y=gate_errors, mode="lines", name="Gate error", yaxis="y2"))
fig.update_layout(
    title="Two-Qubit Calibration Workflow",
    xaxis_title="Sweep parameter",
    yaxis_title="Frequency shift (MHz)",
    yaxis2={"title": "Gate error", "overlaying": "y", "side": "right"},
)
fig.show()

log_and_record("two_qubit_calibration_summary", workflow.results)
print("Two-qubit calibration report")
for key, value in workflow.results.items():
    print(f"  {key}: {value}")